<a href="https://colab.research.google.com/github/jhonyjm4/SERS_GS1/blob/main/SERS_GS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install ollama==0.6.2

import json
from ollama import Client

In [5]:
# ==== CONFIGURAÇÃO DA IA OPERACIONAL (OLLAMA CLOUD) ====

API_KEY = "3a7a5e76c2914495834ec14ab91a4730.ECpPqDEws6dXZ5ZKi9942xUA"
MODEL_NAME = "gpt-oss:120b"

client = Client(
    host="https://ollama.com",
    headers={'Authorization': f'Bearer {API_KEY}'}
)

def chamar_ia_controle_missao(prompt_sistema, prompt_usuario):
    mensagens = [
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": prompt_usuario}
    ]
    try:
        response = client.chat(
            model=MODEL_NAME,
            messages=mensagens,
            options={
                "num_predict": 800,
                "temperature": 0.3
            },
            stream=False
        )
        return response['message']['content'].strip()
    except Exception as e:
        return f"[Alerta de Conexão Ollama: {e} - Acionando Inteligência de Contingência Local]"


# ==== DADOS OPERACIONAIS DA MISSÃO (ALINHADOS AO ESCOPO DE ENERGIA RENOVAÚVEL) ====
nome_missao = "Mission Orion - Sustentabilidade Espacial"
nome_equipe = "Equipe Apollo"
lista_pontos_missao = []

# ==== LIMITES TÉCNICOS E CRITÉRIOS DE RISCO ====
# Temperatura dos Painéis Solares e Baterias (Controle Térmico)
temperatura_critica_maxima = 45
temperatura_critica_minima = -10
temperatura_atencao = 38

# Geração Fotovoltaica (Captação de Energia Renovável)
geracao_critica = 30   # % de eficiência de captação
geracao_atencao = 60

# Armazenamento em Banco de Baterias (Energia Disponível)
bateria_critica = 25   # % de carga restante
bateria_atencao = 55

# Consumo de Potência dos Módulos Operacionais
potencia_critica = 90  # % de sobrecarga do sistema
potencia_atencao = 75

# Eficiência do Inversor/Distribuição Energética
eficiencia_critica = 70 # % de aproveitamento útil
eficiencia_atencao = 85

# ==== MATRIZ DE DADOS SIMULADOS (Escopo Técnico de Energia) ====
# Estrutura de cada ciclo: [Temp. Painéis (°C), Geração Solar (%), Nível Bateria (%), Consumo Potência (%), Eficiência Sistema (%)]
dados_missao = [
    [25.0, 95, 98, 45, 96],  # Ciclo 1: Início nominal
    [32.5, 88, 92, 50, 94],  # Ciclo 2: Operação estável
    [39.0, 55, 75, 78, 88],  # Ciclo 3: Alta temperatura e queda de captação
    [42.2, 28, 48, 88, 82],  # Ciclo 4: Alerta severo de geração e bateria
    [46.5, 15, 20, 95, 65],  # Ciclo 5: Risco crítico / Sobrecarga de Potência
    [35.0, 75, 45, 60, 89]   # Ciclo 6: Recuperação pós-manobra térmica
]

numero_ciclos = len(dados_missao)

areas_monitoradas = [
    "Temperatura dos Painéis",
    "Geração Solar Fotovoltaica",
    "Armazenamento (Baterias)",
    "Consumo de Potência",
    "Eficiência de Distribuição"
]

ciclos_monitorados = [
    "início da missão e calibração solar",
    "estabilização da órbita energética",
    "zona de penumbra / aquecimento térmico",
    "alerta de baixa captação renovável",
    "risco crítico de colapso energético",
    "manobra de reorientação e recuperação"
]

# ==== FUNÇÕES DE ANÁLISE DE RISCO ====
def analisar_temperatura(ciclo):
    val = dados_missao[ciclo][0]
    if val >= temperatura_critica_maxima or val <= temperatura_critica_minima: return 2
    if temperatura_atencao <= val < temperatura_critica_maxima: return 1
    return 0

def analisar_geracao(ciclo):
    val = dados_missao[ciclo][1]
    if val <= geracao_critica: return 2
    if geracao_critica < val <= geracao_atencao: return 1
    return 0

def analisar_bateria(ciclo):
    val = dados_missao[ciclo][2]
    if val <= bateria_critica: return 2
    if bateria_critica < val <= bateria_atencao: return 1
    return 0

def analisar_potencia(ciclo):
    val = dados_missao[ciclo][3]
    if val >= potencia_critica: return 2
    if potencia_atencao <= val < potencia_critica: return 1
    return 0

def analisar_eficiencia(ciclo):
    val = dados_missao[ciclo][4]
    if val <= eficiencia_critica: return 2
    if eficiencia_critica < val <= eficiencia_atencao: return 1
    return 0

# ==== FUNÇÕES DE CÁLCULO ====
def somar_pontos_risco_ciclo(ciclo):
    return sum([
        analisar_temperatura(ciclo),
        analisar_geracao(ciclo),
        analisar_bateria(ciclo),
        analisar_potencia(ciclo),
        analisar_eficiencia(ciclo)
    ])

def calc_media_area_missao(info):
    soma = sum(dados_missao[i][info] for i in range(numero_ciclos))
    return soma / numero_ciclos

def calc_media_risco_missao():
    soma = sum(somar_pontos_risco_ciclo(i) for i in range(numero_ciclos))
    return soma / numero_ciclos

# ==== FUNÇÕES AUXILIARES DE CLASSIFICAÇÃO ====
def pontuacao_para_status(pontos):
    if pontos == 0: return "NORMAL"
    if pontos == 1: return "ATENÇÃO"
    return "CRÍTICO"

def listar_pontos_areas(ciclo):
    return [
        analisar_temperatura(ciclo),
        analisar_geracao(ciclo),
        analisar_bateria(ciclo),
        analisar_potencia(ciclo),
        analisar_eficiencia(ciclo)
    ]

def classificar_ciclo(ciclo):
    pontos = somar_pontos_risco_ciclo(ciclo)
    if pontos <= 2: return "SISTEMA ENERGÉTICO ESTÁVEL"
    if pontos <= 5: return "ALERTA: FLUTUAÇÃO ENERGÉTICA"
    return "RESTRUTURAÇÃO DE CARGA CRÍTICA"

def classificar_missao(risco_medio):
    if risco_medio >= 6: return "OPERANDO EM LIMITE CRÍTICO SUSTENTÁVEL"
    if 3 <= risco_medio <= 5: return "MATRIZ ENERGÉTICA EM ATENÇÃO"
    return "GESTÃO ENERGÉTICA NOMINAL SUSTENTÁVEL"

def identificar_area_mais_afetada_ciclo(ciclo):
    pontos_area = listar_pontos_areas(ciclo)
    maior_pontuacao = max(pontos_area)
    if maior_pontuacao == 0: return "Nenhum gargalo energético"

    indices = [i for i, x in enumerate(pontos_area) if x == maior_pontuacao]
    areas = [areas_monitoradas[idx] for idx in indices]

    if len(areas) == 1: return areas[0]
    return ", ".join(areas[:-1]) + " e " + areas[-1]

def identificar_ciclo_mais_critico():
    lista_riscos = [somar_pontos_risco_ciclo(i) for i in range(numero_ciclos)]
    maior_risco = max(lista_riscos)
    indices = [i for i, x in enumerate(lista_riscos) if x == maior_risco]
    ciclos = [str(idx + 1) for idx in indices]

    if len(ciclos) == 1: return ciclos[0]
    return ", ".join(ciclos[:-1]) + " e " + ciclos[-1]

def numero_ciclos_criticos():
    return sum(1 for i in range(numero_ciclos) if somar_pontos_risco_ciclo(i) > 5)

def analisar_tendencia_ciclo(ciclo):
    if ciclo == 0: return "Sem histórico"
    atual = somar_pontos_risco_ciclo(ciclo)
    anterior = somar_pontos_risco_ciclo(ciclo - 1)
    if atual == anterior: return "Estável"
    return "Piora no balanço térmico/energético" if atual > anterior else "Melhora na eficiência"

# ==== INTEGRAÇÃO INTELIGENTE DE RECOMENDAÇÕES (OLLAMA) ====
def gerar_recomendacao(ciclo):
    recomendacoes = []
    if analisar_temperatura(ciclo) == 2: recomendacoes.append("acionar resfriamento dos painéis")
    if analisar_geracao(ciclo) == 2: recomendacoes.append("reorientar matriz fotovoltaica para o vetor solar")
    if analisar_bateria(ciclo) == 2: recomendacoes.append("cortar linhas de consumo não-essenciais e preservar baterias")
    if analisar_potencia(ciclo) == 2: recomendacoes.append("limitar picos de potência nos subsistemas")
    if analisar_eficiencia(ciclo) == 2: recomendacoes.append("isolar perdas térmicas no inversor principal")

    if not recomendacoes:
        texto_contingencia = "Manter protocolo verde de operação"
    else:
        texto_contingencia = ", ".join(recomendacoes[:-1]) + " e " + recomendacoes[-1] if len(recomendacoes) > 1 else recomendacoes[0]

    if somar_pontos_risco_ciclo(ciclo) == 0:
        return "Matriz 100% renovável ativa. Sem anomalias registradas."

    prompt_sistema = (
        "Você é o Engenheiro Chefe da IA de gerenciamento energético da nave Orion. "
        "Com base na telemetria, gere ordens curtas, puramente voltadas à conservação de energia e sustentabilidade de hardware."
    )

    prompt_usuario = (
        f"Alerta de Energia no Ciclo {ciclo + 1}: {ciclos_monitorados[ciclo].upper()}.\n"
        f"Métricas de Sustentabilidade:\n"
        f"- Temperatura Painéis: {dados_missao[ciclo][0]} °C\n"
        f"- Eficiência de Captação: {dados_missao[ciclo][1]}%\n"
        f"- Carga das Baterias: {dados_missao[ciclo][2]}%\n"
        f"- Carga de Potência Exigida: {dados_missao[ciclo][3]}%\n"
        f"- Aproveitamento do Inversor: {dados_missao[ciclo][4]}%\n"
        f"Diretriz técnica local: {texto_contingencia}.\n"
        f"Formule o comando operacional emergencial."
    )

    resposta_ia = chamar_ia_controle_missao(prompt_sistema, prompt_usuario)
    if "[Alerta" in resposta_ia:
        return f"{texto_contingencia} (Modo de Contingência de Energia Local Ativo)"
    return resposta_ia


def gerar_relatorio_final():
    soma_temp = sum(analisar_temperatura(i) for i in range(numero_ciclos))
    soma_ger = sum(analisar_geracao(i) for i in range(numero_ciclos))
    soma_bat = sum(analisar_bateria(i) for i in range(numero_ciclos))
    soma_pot = sum(analisar_potencia(i) for i in range(numero_ciclos))
    soma_efi = sum(analisar_eficiencia(i) for i in range(numero_ciclos))

    risco_inicial = somar_pontos_risco_ciclo(0)
    risco_final = somar_pontos_risco_ciclo(numero_ciclos-1)
    tendencia = "Estável" if risco_final == risco_inicial else ("Piora Energética" if risco_final > risco_inicial else "Recuperação Sustentável")

    pontuacoes = [soma_temp, soma_ger, soma_bat, soma_pot, soma_efi]
    maior_pontuacao = max(pontuacoes)
    areas_afetadas = [areas_monitoradas[idx] for idx, pts in enumerate(pontuacoes) if pts == maior_pontuacao]
    texto_areas = ", ".join(areas_afetadas[:-1]) + " e " + areas_afetadas[-1] if len(areas_afetadas) > 1 else areas_afetadas[0]

    print("\n" + "="*46 + " RELATÓRIO FINAL DE SUSTENTABILIDADE " + "="*45)
    print(f"Missão: {nome_missao} | Equipe: {nome_equipe}")
    print(f"Ciclos Elétricos Analisados: {numero_ciclos}")
    print(f"Gargalo Acumulado Principal: {texto_areas} ({maior_pontuacao} pontos de instabilidade acumulados)")
    print(f"- Vetor de Tendência Geral: {tendencia}")
    print(f"- Risco Inicial: {risco_inicial} pts | Risco Final: {risco_final} pts")
    print("\n-- Balanço de Risco Acumulado por Subsistema --")
    print(f"- Estabilidade Térmica dos Painéis: {soma_temp} pts")
    print(f"- Eficiência da Captação Renovável: {soma_ger} pts")
    print(f"- Vida Útil e Carga de Baterias: {soma_bat} pts")
    print(f"- Sobrecarga por Demanda de Potência: {soma_pot} pts")
    print(f"- Eficiência de Conversão do Inversor: {soma_efi} pts")
    print(f"\n- Ciclo Mais Crítico do Sistema: Ciclo {identificar_ciclo_mais_critico()} (Pico: {max([somar_pontos_risco_ciclo(i) for i in range(numero_ciclos)])} pts)")
    print(f"- Média Geral de Risco da Matriz: {calc_media_risco_missao():.2f}")

    print("\n-- Médias de Telemetria de Hardware --")
    print(f"- Temperatura Média Operacional: {calc_media_area_missao(0):.2f} ºC")
    print(f"- Eficiência Média Fotovoltaica: {calc_media_area_missao(1):.2f}%")
    print(f"- Nível Médio de Armazenamento: {calc_media_area_missao(2):.2f}%")
    print(f"- Potência Média Requerida: {calc_media_area_missao(3):.2f}%")
    print(f"- Eficiência Sistêmica Média: {calc_media_area_missao(4):.2f}%")

    print(f"\nClassificação de Engenharia da Missão: {classificar_missao(calc_media_risco_missao())}")
    print("-" * 128)

    # ==== PARECER COGNITIVO DA IA VOLTADO A SUSTENTABILIDADE (REQUISITO DA GLOBAL SOLUTION) ====
    print("PARECER COGNITIVO DA IA (ANÁLISE DE SUSTENTABILIDADE ENERGÉTICA ESPACIAL):")
    prompt_sistema_relatorio = (
        "Você é o Diretor de Sustentabilidade e Infraestrutura de Energia da Agência Apollo. "
        "Analise o histórico completo de telemetria da missão e elabore um parecer técnico descritivo de até 4 linhas "
        "propondo melhorias em eficiência energética e uso de recursos renováveis para as próximas fases."
    )
    prompt_usuario_relatorio = (
        f"Métricas Consolidadas da Missão Orion:\n"
        f"- Classificação de Operação: {classificar_missao(calc_media_risco_missao())}\n"
        f"- Áreas de Maior Desgaste Acumulado: {texto_areas}\n"
        f"- Eficiência Média da Energia: {calc_media_area_missao(2):.2f}%\n"
        f"- Estabilidade Média dos Módulos: {calc_media_area_missao(4):.2f}%\n"
        f"Gere uma análise preditiva e focada em sustentabilidade de hardware."
    )
    conclusao_ia = chamar_ia_controle_missao(prompt_sistema_relatorio, prompt_usuario_relatorio)
    print(conclusao_ia)
    print("="*128)


# ==== EXECUÇÃO DO FLUXO PRINCIPAL ====
print("="*128)
print("                                       SISTEMA INTELIGENTE DE TELEMETRIA ENERGÉTICA (SERS)")
print("="*128)
print(f"Missão Ativa: {nome_missao} | Operador: {nome_equipe}\n")

for i in range(numero_ciclos):
    lista_pontos_missao.append(somar_pontos_risco_ciclo(i))
    print(f"--- CICLO {i+1}: {ciclos_monitorados[i].upper()} ---")
    print(f"  > Temp. Painéis: {dados_missao[i][0]} °C | Status: {pontuacao_para_status(analisar_temperatura(i))}")
    print(f"  > Geração Solar: {dados_missao[i][1]} %   | Status: {pontuacao_para_status(analisar_geracao(i))}")
    print(f"  > Nível Bateria: {dados_missao[i][2]} %   | Status: {pontuacao_para_status(analisar_bateria(i))}")
    print(f"  > Demanda Potência: {dados_missao[i][3]} %| Status: {pontuacao_para_status(analisar_potencia(i))}")
    print(f"  > Eficiência Inversor: {dados_missao[i][4]} % | Status: {pontuacao_para_status(analisar_eficiencia(i))}")
    print(f"  * Tendência Recente: {analisar_tendencia_ciclo(i)}")
    print(f"  * Risco do Ciclo: {somar_pontos_risco_ciclo(i)} / 10 | Classificação: {classificar_ciclo(i)}")
    print(f"  * Gargalo Atual: {identificar_area_mais_afetada_ciclo(i)}")
    print(f"  [Ordem Recomendada pela IA]: {gerar_recomendacao(i)}\n")

gerar_relatorio_final()

                                       SISTEMA INTELIGENTE DE TELEMETRIA ENERGÉTICA (SERS)
Missão Ativa: Mission Orion - Sustentabilidade Espacial | Operador: Equipe Apollo

--- CICLO 1: INÍCIO DA MISSÃO E CALIBRAÇÃO SOLAR ---
  > Temp. Painéis: 25.0 °C | Status: NORMAL
  > Geração Solar: 95 %   | Status: NORMAL
  > Nível Bateria: 98 %   | Status: NORMAL
  > Demanda Potência: 45 %| Status: NORMAL
  > Eficiência Inversor: 96 % | Status: NORMAL
  * Tendência Recente: Sem histórico
  * Risco do Ciclo: 0 / 10 | Classificação: SISTEMA ENERGÉTICO ESTÁVEL
  * Gargalo Atual: Nenhum gargalo energético
  [Ordem Recomendada pela IA]: Matriz 100% renovável ativa. Sem anomalias registradas.

--- CICLO 2: ESTABILIZAÇÃO DA ÓRBITA ENERGÉTICA ---
  > Temp. Painéis: 32.5 °C | Status: NORMAL
  > Geração Solar: 88 %   | Status: NORMAL
  > Nível Bateria: 92 %   | Status: NORMAL
  > Demanda Potência: 50 %| Status: NORMAL
  > Eficiência Inversor: 94 % | Status: NORMAL
  * Tendência Recente: Estável
  * Risco